## Imports


In [ ]:
from datasets import load_dataset
from collections import defaultdict, Counter
import re
import math
import pickle

## Load dataset

In [ ]:
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

In [ ]:
training_set=ds['train']['text']
testing_set=ds['test']['text']
print(f"Sample: {training_set[4]}")
print(f"train_size = {len(training_set)}")
print(f"test_size = {len(testing_set)}")

Sample:  The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomers . Character designer Raita Honjou and composer Hitoshi Sakimoto both returned from previous entries , along with Valkyria Chronicles II director Takeshi Ozawa . A large team of writers handled the script . The game 's opening theme was sung by May 'n . 

train_size = 36718
test_size = 4358


## Functions

In [ ]:
def preprocess(texts):
  """
  preprocess data -> lowercasing and remove punctuation
  Tokenize and return corpus
  """
  corpus = []
  for text in texts:
      if text.strip() == "":
          continue
      text = text.lower()
      text = re.sub(r"[^a-z0-9\s]", "", text)
      words = text.split()
      if words:
          corpus.append(words)
  return corpus

def compute_perplexity(model,corpus):
  """
  ppl(S) = e^x where x = -(1/n) * log(P(w₁, ..., wₙ)) = -(1/n) * ∑log(P(wᵢ|w₁...wᵢ₋₁))
  """
  sum_logProb = 0.0
  N = 0
  for sentence in corpus:
      N += len(sentence)
      sentence = ["<start>"] * (model.n - 1) + sentence + ["</end>"]
      for i in range(len(sentence) - model.n + 1):
          ngram = tuple(sentence[i:i+model.n])
          p = model.compute_probability(ngram)
          sum_logProb += math.log(p)
  return math.exp(-sum_logProb / N)

## N-gram Language Model

In [ ]:
class LaplaceLanguageModel:
    def __init__(self, n=1):
        self.n = n
        self.ngram_counts = defaultdict(int)
        self.context_counts = defaultdict(int)
        self.vocabulary = set()
        self.V = 0

    def train(self, corpus):
        for sentence in corpus:
            sentence = ["<start>"] * (self.n - 1) + sentence + ["</end>"]
            for i in range(len(sentence) - self.n + 1):
                ngram = tuple(sentence[i:i+self.n])
                context = tuple(sentence[i:i+self.n-1])
                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1
                self.vocabulary.update(ngram)
        self.V = len(self.vocabulary)
        print(f"Training complete: {len(self.ngram_counts)} n-grams, vocabulary size = {self.V}")

    def compute_probability(self, ngram,alpha=1):
      """
      compute probability with (add-alpha) laplace smoothing to account for unseen ngrams
      """
      context = ngram[:-1]
      ngram_count = self.ngram_counts[ngram]
      context_count = self.context_counts[context]
      return (ngram_count + alpha) / (context_count + alpha*self.V)

    def predict_next_word(self, context):
      if self.n == 1:
        context = ()
      else:
        context_tokens = context[-(self.n - 1):]
        if len(context_tokens) < self.n - 1:
          context_tokens = ["<start>"] * (self.n - 1 - len(context_tokens)) + context_tokens
        context = tuple(context_tokens)

      best_word = None
      best_prob = -1
      for word in self.vocabulary:
        ngram = context + (word,)
        prob  = self.compute_probability(ngram)
        if prob > best_prob:
            best_prob = prob
            best_word = ngram[-1]
      return best_word

    def save(self, path='ngram_model.pkl'):
        with open(path, 'wb') as f:
            pickle.dump({
                'n':              self.n,
                'ngram_counts':   dict(self.ngram_counts),
                'context_counts': dict(self.context_counts),
                'vocabulary':     self.vocabulary,
                'V':              self.V,
            }, f)
        print(f"Model saved to {path}")

## Prepare Data

In [ ]:
train_corpus = preprocess(training_set)
test_corpus  = preprocess(testing_set)
print(f"Train sentences: {len(train_corpus)}")
print(f"Test  sentences: {len(test_corpus)}")

Train sentences: 23764
Test  sentences: 2891


## Unigram Model

In [ ]:
unigram_model = LaplaceLanguageModel(n=1)
unigram_model.train(train_corpus)
unigram_ppl = compute_perplexity(unigram_model, test_corpus)
print(f"Unigram Perplexity: {unigram_ppl:.2f}")

Training complete: 65333 n-grams, vocabulary size = 65333
Unigram Perplexity: 2478.66


## Comparing different n-values and saving best model

In [ ]:
n_values = [2, 3, 4]
perplexities = []
best_perplexity = float("inf")
best_n = None
best_model = None

for n in n_values:
    model = LaplaceLanguageModel(n=n)
    model.train(train_corpus)

    perplexity = compute_perplexity(model, test_corpus)
    perplexities.append(perplexity)

    print(f"Perplexity for N = {n}: {perplexity:.2f}\n")

    if perplexity < best_perplexity:
        best_perplexity = perplexity
        best_n = n
        best_model = model

best_model.save("best_model.pkl")
print(f"\nBest model saved: {best_n}-gram with perplexity {best_perplexity:.2f}")

Training complete: 703280 n-grams, vocabulary size = 65334
Perplexity for N = 2: 10494.56

Training complete: 1364029 n-grams, vocabulary size = 65334
Perplexity for N = 3: 45987.55

Training complete: 1626356 n-grams, vocabulary size = 65334
Perplexity for N = 4: 65772.29

Model saved to best_model.pkl

Best model saved: 2-gram with perplexity 10494.56


In [ ]:
print(f"\n{'Model':<10} {'Perplexity':>12}")
print("-" * 23)
print(f"{'Unigram':<10} {unigram_ppl:>12.2f}")
for i, n in enumerate(n_values):
    print(f"{str(n)+'-gram':<10} {perplexities[i]:>12.2f}")


Model        Perplexity
-----------------------
Unigram         2478.66
2-gram         10494.56
3-gram         45987.55
4-gram         65772.29


## Infere on best model

In [ ]:
def run_inference(path, prompt, max_tokens=20):
    """
    loads saved n-gram model, preprocesses input prompt and predicts next tokens
    """
    loaded_model = load_model(path)
    cleaned = preprocess([prompt])
    tokens  = cleaned[0]
    for _ in range(max_tokens):
        next_word = loaded_model.predict_next_word(tokens)
        if next_word is None or next_word == "</end>":
            break
        tokens.append(next_word)
    return tokens

def load_model(path="ngram_model.pkl"):
    with open(path, "rb") as f:
        data = pickle.load(f)
    model= LaplaceLanguageModel(n=data["n"])
    model.ngram_counts = defaultdict(int, data["ngram_counts"])
    model.context_counts = defaultdict(int, data["context_counts"])
    model.vocabulary = data["vocabulary"]
    model.V = data["V"]
    print(f"Model loaded from {path}")
    return model

In [ ]:
prompt = "The! game began"
output = run_inference(f"best_model.pkl", prompt, max_tokens=30)
print(f"Prompt : {prompt}")
print(f"Tokens: {output}")
print(f"Generated Sentence: { ' '.join(output)}")

Model loaded from best_model.pkl
Prompt : The! game began
Tokens: ['the', 'game', 'began', 'to', 'the', 'first', 'time', 'the', 'first', 'time', 'the', 'first', 'time', 'the', 'first', 'time', 'the', 'first', 'time', 'the', 'first', 'time', 'the', 'first', 'time', 'the', 'first', 'time', 'the', 'first', 'time', 'the', 'first']
Generated Sentence: the game began to the first time the first time the first time the first time the first time the first time the first time the first time the first time the first
